# NeuroNav FPGA Digital Twin Simulation Colab


## Purpose

This notebook is the PC-based digital twin plan for NeuroNav. It simulates fixed-point LIF inference and prepares test vectors for RTL verification. It is valid when no physical FPGA is available, as long as all hardware metrics are labeled as simulated or estimated.

In [ ]:
!pip -q install numpy matplotlib pandas scikit-learn

In [ ]:
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('/content/neuro_nav_digital_twin')
OUT.mkdir(exist_ok=True)
print(OUT)

## Fixed-point LIF equation

Use one integer equation as the shared contract between Python and RTL:

```text
mem_next = ((beta_q * mem) >> beta_frac_bits) + weighted_spike_sum
spike = mem_next >= threshold
mem_next = reset_value when spike else mem_next
```

The exact bit widths must be chosen after measuring trained weight ranges and membrane ranges.

In [ ]:
def lif_step_int(mem, input_current, beta_q=243, beta_frac_bits=8, threshold=128, reset=0):
    mem_next = ((beta_q * mem) >> beta_frac_bits) + input_current
    spike = mem_next >= threshold
    mem_next = np.where(spike, reset, mem_next)
    return mem_next.astype(np.int32), spike.astype(np.int8)

# Smoke test for one neuron over synthetic current.
mem = np.array([0], dtype=np.int32)
trace = []
for cur in [20, 30, 40, 50, 20, 10, 120, 5]:
    mem, spk = lif_step_int(mem, np.array([cur], dtype=np.int32))
    trace.append({'input_current': cur, 'membrane': int(mem[0]), 'spike': int(spk[0])})
pd.DataFrame(trace)

In [ ]:
def estimate_time_multiplexed_cycles(num_inputs=784, hidden=128, outputs=10, time_steps=25, macs_per_cycle=1):
    # Conservative dense schedule. Sparse/event-driven optimization should be measured separately.
    layer1_cycles = math.ceil(num_inputs * hidden / macs_per_cycle)
    layer2_cycles = math.ceil(hidden * outputs / macs_per_cycle)
    cycles_per_step = layer1_cycles + layer2_cycles
    cycles_per_inference = cycles_per_step * time_steps
    return {
        'num_inputs': num_inputs,
        'hidden_neurons': hidden,
        'outputs': outputs,
        'time_steps': time_steps,
        'macs_per_cycle': macs_per_cycle,
        'cycles_per_step': cycles_per_step,
        'cycles_per_inference': cycles_per_inference,
    }

rows = []
for macs in [1, 4, 8, 16, 32, 64, 128]:
    row = estimate_time_multiplexed_cycles(macs_per_cycle=macs)
    for clk_mhz in [25, 50, 100]:
        row2 = dict(row)
        row2['clock_mhz'] = clk_mhz
        row2['latency_ms_est'] = row['cycles_per_inference'] / (clk_mhz * 1e6) * 1000
        row2['throughput_ips_est'] = (clk_mhz * 1e6) / row['cycles_per_inference']
        rows.append(row2)
df = pd.DataFrame(rows)
df.head(12)

In [ ]:
# Memory estimate for a 784-128-10 dense SNN.
def memory_estimate(num_inputs=784, hidden=128, outputs=10, weight_bits=8, state_bits=16):
    weights = num_inputs * hidden + hidden * outputs
    neurons = hidden + outputs
    return {
        'weights': weights,
        'weight_kbits': weights * weight_bits / 1024,
        'neurons': neurons,
        'state_kbits': neurons * state_bits / 1024,
        'total_kbits_est': (weights * weight_bits + neurons * state_bits) / 1024,
    }
mem_est = memory_estimate()
mem_est

In [ ]:
# Generate deterministic RTL test vectors for a LIF neuron.
rng = np.random.default_rng(42)
input_currents = rng.integers(low=-16, high=80, size=64, dtype=np.int32)
mem = np.array([0], dtype=np.int32)
rows = []
for cycle, cur in enumerate(input_currents):
    old_mem = int(mem[0])
    mem, spk = lif_step_int(mem, np.array([cur], dtype=np.int32))
    rows.append({'cycle': cycle, 'input_current': int(cur), 'mem_in': old_mem, 'mem_out': int(mem[0]), 'spike': int(spk[0])})
tv = pd.DataFrame(rows)
tv.to_csv(OUT / 'lif_neuron_test_vectors.csv', index=False)
print(tv.head())
print('saved', OUT / 'lif_neuron_test_vectors.csv')

In [ ]:
verilog = r'''
module lif_neuron #(
    parameter integer BETA_Q = 243,
    parameter integer BETA_FRAC_BITS = 8,
    parameter integer THRESHOLD = 128,
    parameter integer RESET_VALUE = 0
)(
    input  wire clk,
    input  wire rst,
    input  wire signed [31:0] input_current,
    output reg  signed [31:0] membrane,
    output reg spike
);
    reg signed [63:0] leaked;
    reg signed [31:0] mem_next;

    always @(posedge clk) begin
        if (rst) begin
            membrane <= 0;
            spike <= 0;
        end else begin
            leaked = (BETA_Q * membrane) >>> BETA_FRAC_BITS;
            mem_next = leaked[31:0] + input_current;
            if (mem_next >= THRESHOLD) begin
                membrane <= RESET_VALUE;
                spike <= 1'b1;
            end else begin
                membrane <= mem_next;
                spike <= 1'b0;
            end
        end
    end
endmodule
'''
(OUT / 'lif_neuron.v').write_text(verilog)
print('saved', OUT / 'lif_neuron.v')

## RTL verification route

1. Use `lif_neuron_test_vectors.csv` as the golden reference.
2. Simulate `lif_neuron.v` with cocotb plus Verilator/Icarus.
3. Compare every cycle's `membrane` and `spike` output against the CSV.
4. Only after exact match, scale to synapse accumulation and small network inference.

Until a real FPGA board is available, resource, timing, and power must be labeled as estimated or simulated.